# 03 · Gold – H2R Star Schema
**Layer:** Gold (business-ready) · Consumed by the Direct Lake semantic model

| Dimensions | Facts |
|---|---|
| `dim_date`, `dim_employee`, `dim_department`, `dim_location`, `dim_grade`, `dim_course` | `fact_headcount_monthly` (snapshot), `fact_recruitment`, `fact_onboarding`, `fact_training`, `fact_leave`, `fact_performance`, `fact_payroll`, `fact_exit` |

Every fact joins to dimensions on surrogate-free business keys plus an integer `date_key (yyyymmdd)`.

In [ ]:
from pyspark.sql import functions as F

def save(df, name):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(name)
    print(f"✔ {name:<28} {spark.table(name).count():>8,} rows")

dk = lambda c: F.date_format(c, "yyyyMMdd").cast("int")

## Dimensions

In [ ]:
# Date dimension with Indian fiscal year (Apr–Mar)
dates = spark.sql("SELECT explode(sequence(to_date('2021-01-01'), to_date('2027-12-31'), interval 1 day)) AS date")
dim_date = (dates
    .withColumn("date_key", dk("date"))
    .withColumn("year", F.year("date")).withColumn("month_no", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMM"))
    .withColumn("month_start", F.trunc("date", "month"))
    .withColumn("year_month", F.date_format("date", "yyyy-MM"))
    .withColumn("fiscal_year", F.concat(F.lit("FY"), F.date_format(F.add_months("date", 9), "yy")))
    .withColumn("fiscal_quarter", F.concat(F.lit("Q"), F.quarter(F.add_months("date", -3))))
    .withColumn("is_month_end", F.col("date") == F.last_day("date")))
save(dim_date, "dim_date")

save(spark.table("silver_department"), "dim_department")
save(spark.table("silver_location"), "dim_location")
save(spark.table("silver_grade"), "dim_grade")
save(spark.table("silver_course"), "dim_course")

save(spark.table("silver_employee").select(
        "employee_id", "full_name", "gender", "age_band", "tenure_band", "tenure_years",
        "department_id", "location_id", "grade_id", "employment_type", "work_mode",
        "manager_id", "hire_date", "termination_date", "employment_status", "is_active",
        "candidate_id", "email"),
     "dim_employee")

## Monthly headcount snapshot (the backbone of attrition maths)

In [ ]:
months = dim_date.filter("is_month_end").select(F.col("date").alias("month_end"), "date_key")
emp = spark.table("silver_employee")

fact_hc = (months.crossJoin(emp.select("employee_id", "department_id", "location_id", "grade_id",
                                       "gender", "hire_date", "termination_date"))
    .filter((F.col("hire_date") <= F.col("month_end")) &
            (F.col("termination_date").isNull() | (F.col("termination_date") >= F.trunc("month_end", "month"))))
    .filter(F.col("month_end") <= F.last_day(F.current_date()))
    .withColumn("is_new_hire",  F.trunc("hire_date", "month") == F.trunc("month_end", "month"))
    .withColumn("is_exit",      F.trunc("termination_date", "month") == F.trunc("month_end", "month"))
    .withColumn("is_month_end_active", F.col("termination_date").isNull() | (F.col("termination_date") > F.col("month_end")))
    .select("date_key", "month_end", "employee_id", "department_id", "location_id", "grade_id",
            "is_new_hire", "is_exit", "is_month_end_active"))
save(fact_hc, "fact_headcount_monthly")

## Lifecycle facts: Hire → Onboard → Develop → Perform → Pay → Retire

In [ ]:
cand = spark.table("silver_candidate")
req  = spark.table("silver_requisition")
hires = emp.select("candidate_id", "employee_id", "hire_date").filter("candidate_id IS NOT NULL")

save(cand.join(req.select("requisition_id", "department_id", "location_id", "grade_id", "opened_date"), "requisition_id", "left")
         .join(hires, "candidate_id", "left")
         .withColumn("date_key", dk("applied_date"))
         .withColumn("is_hired", F.col("current_stage") == "Hired")
         .withColumn("is_offered", F.col("stage_order") >= 4)
         .withColumn("time_to_hire_days", F.datediff("hire_date", "applied_date"))
         .withColumn("time_to_fill_days", F.datediff("hire_date", "opened_date")),
     "fact_recruitment")

save(spark.table("silver_onboarding").withColumn("date_key", dk("onboarding_start")), "fact_onboarding")

save(spark.table("silver_training")
         .withColumn("date_key", dk("enrolled_date"))
         .withColumn("is_completed", F.col("completion_status") == "Completed")
         .join(spark.table("silver_course").select("course_id", "duration_hours"), "course_id", "left")
         .withColumn("learning_hours", F.when(F.col("is_completed"), F.col("duration_hours")).otherwise(0)),
     "fact_training")

save(spark.table("silver_leave").withColumn("date_key", dk("leave_start_date")), "fact_leave")
save(spark.table("silver_performance").withColumn("date_key", dk("fiscal_year_end")), "fact_performance")
save(spark.table("silver_payroll").withColumn("date_key", dk("pay_month")), "fact_payroll")

save(spark.table("silver_exit")
         .join(emp.select("employee_id", "department_id", "location_id", "grade_id", "hire_date"), "employee_id", "left")
         .withColumn("date_key", dk("last_working_date"))
         .withColumn("tenure_at_exit_years", F.round(F.months_between("last_working_date", "hire_date") / 12, 1)),
     "fact_exit")

In [ ]:
spark.sql("SHOW TABLES").filter("tableName LIKE 'dim_%' OR tableName LIKE 'fact_%'").show(50, False)